# 1. Data Preparation

This stage aims to create image tiles by slicing the remote sensing raster dataset into a specified size. The resulting tiles are used for annotation and labeling of **oil palm tree objects** on the Roboflow platform. These annotated images will serve as the training data for the **YOLOv26** object detection model.


In [7]:
# Use this code to mount your Google Drive in Google Colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics
!pip install torch torchvision
!pip install opencv-python
!pip install pillow
!pip install matplotlib
!pip install geopandas
!pip install gdal
!pip install patchify
!pip install rasterio
!pip install shapely
!pip install fiona
!pip install tifffile
!pip install roboflow
!pip install scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 974.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 38.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
patchify 0.2.3 requires numpy<2,>=1, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


## Convert to 8-bit Raster Format

YOLOv26 by default operates with **8-bit input image format (uint8)**, although it does not explicitly restrict inputs to this format.  
However, the pretrained weights and default augmentations are designed with this assumption for standard RGB images.  

Therefore, the raster images are converted to 8-bit format to ensure compatibility with the standard inference pipeline provided by **Ultralytics**.  
Additionally, most public datasets (such as **COCO**) and common deployment frameworks (**PyTorch**, **OpenCV**) assume **RGB 8-bit** input images.


### Check Raster Format

This step is used to verify the format of the raster data to ensure that it is already in **uint8** format, or to identify if it is still in another data type before conversion.

In [1]:
import rasterio

# Path to the original raster data
image_path = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data/raster_data_3.tif'

# Opening the raster file
with rasterio.open(image_path) as src:
    # Getting basic metadata information
    print("Metadata Information:")
    print(f"- CRS (Coordinate Reference System): {src.crs}")
    print(f"- Dimensions (Height, Width): {src.height}, {src.width}")
    print(f"- Number of Bands: {src.count}")
    print(f"- Data Type: {src.dtypes}")
    print(f"- No-Data Value (nodata): {src.nodata}")
    print(f"- Geo Transform: {src.transform}")

    # Checking statistics for each band
    print("\nStatistics Per Band:")
    for i in range(1, src.count + 1):
        band_data = src.read(i)  # Reading data for each band
        print(f"  Band {i}: Min={band_data.min()}, Max={band_data.max()}, Mean={band_data.mean()}")

    # Displaying data dimensions per band
    print(f"\nData Dimensions per Band: {src.read(1).shape}")

Metadata Information:
- CRS (Coordinate Reference System): EPSG:32647
- Dimensions (Height, Width): 11817, 5292
- Number of Bands: 3
- Data Type: ('float32', 'float32', 'float32')
- No-Data Value (nodata): -32767.0
- Geo Transform: | 0.09, 0.00, 745892.08|
| 0.00,-0.09, 96827.36|
| 0.00, 0.00, 1.00|

Statistics Per Band:
  Band 1: Min=-32767.0, Max=0.3191177248954773, Mean=-8061.62744140625
  Band 2: Min=-32767.0, Max=0.5161193013191223, Mean=-8061.5966796875
  Band 3: Min=-32767.0, Max=0.5366886854171753, Mean=-8061.62158203125

Data Dimensions per Band: (11817, 5292)


### Convert Raster Format

Create a function to execute raster data conversion into **uint8** format.

In [2]:
import rasterio
import numpy as np
import cv2
import os

# Directory path to input and output raster data
input_dir = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data/'
output_dir = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data/'
input_path = f'{input_dir}/raster_data_3.tif'
output_path = f'{output_dir}/raster_data_3_uint8.tif' # Output file path

# Brightness, Contrast, and Gamma Factor
brightness_factor = 30   # Add brightness value (0-255)
contrast_factor = 1      # Increase contrast (1.0 = original, >1 for brighter)
gamma_value = 1.2        # Gamma correction (1.0 = original, <1 darker, >1 brighter)

# Function for gamma correction
def gamma_correction(image, gamma):
    inv_gamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(image, table)

# Check if input file exists
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Input file not found: {input_path}")

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Opening the input raster file
with rasterio.open(input_path) as src:
    # Log processing information
    print(f"Processing: {os.path.basename(input_path)}")
    print(f"Bands: {src.count}, Size: {src.height}x{src.width}")
    print(f"Settings - Brightness: {brightness_factor}, Contrast: {contrast_factor}, Gamma: {gamma_value}")

    # Reading only the first 3 bands
    image_data = src.read([1, 2, 3])  # Original float32 data

    # Handling no-data values
    nodata = src.nodata
    if nodata is not None:
        image_data = np.where(image_data == nodata, np.nan, image_data)

    # Normalizing values to 0-255 range using percentile stretch (better for remote sensing)
    min_val = np.nanpercentile(image_data, 2, axis=(1, 2), keepdims=True)
    max_val = np.nanpercentile(image_data, 98, axis=(1, 2), keepdims=True)
    scale_range = max_val - min_val
    scale_range[scale_range == 0] = 1  # Avoid division by zero
    image_normalized = ((image_data - min_val) / scale_range * 255).astype(np.float32)

    # Brightness and contrast adjustment
    image_adjusted = (image_normalized * contrast_factor + brightness_factor).clip(0, 255).astype(np.uint8)

    # Apply gamma correction to each band
    image_gamma_corrected = np.zeros_like(image_adjusted, dtype=np.uint8)
    for i in range(3):  # R, G, B bands
        image_gamma_corrected[i] = gamma_correction(image_adjusted[i], gamma_value)

    # Handling NaN by replacing No-Data values with 0
    image_gamma_corrected = np.nan_to_num(image_gamma_corrected, nan=0).astype(np.uint8)

    # Copying metadata and updating output file information
    out_meta = src.meta.copy()
    out_meta.update({
        'dtype': 'uint8',   # uint8 data format
        'count': 3,         # Only 3 RGB bands
        'nodata': 0         # No-data set to 0
    })

    # Writing the adjusted raster file
    with rasterio.open(output_path, 'w', **out_meta) as dest:
        dest.write(image_gamma_corrected)

print("raster_data Conversion with Brightness, Contrast, and Gamma Correction adjustment successful:", output_path)

Processing: raster_data_3.tif
Bands: 3, Size: 11817x5292
Settings - Brightness: 30, Contrast: 1, Gamma: 1.2


/tmp/ipykernel_2663/1034470423.py:53: RuntimeWarning: invalid value encountered in cast
  image_adjusted = (image_normalized * contrast_factor + brightness_factor).clip(0, 255).astype(np.uint8)


raster_data Conversion with Brightness, Contrast, and Gamma Correction adjustment successful: /content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data//raster_data_3_uint8.tif


## Raster Tiling

This stage is used to generate image tiles from raster data that have been converted into **uint8** format, producing output images in `.png` format. The tile size is set according to the **YOLOv26** standard, which is **640x640 pixels**. The resulting images will then be used as input for the annotation or labeling process in **Roboflow**.


In [3]:
import numpy as np
from patchify import patchify
import tifffile as tiff
import os
from PIL import Image

# Paths to your images
original_path = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data'
image_path = f'{original_path}/raster_data_3_uint8.tif'
tiles_path = r'/content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data/tiles/png/3'

# Create directory for tiles if it doesn't exist
os.makedirs(tiles_path, exist_ok=True)

# Load images
large_image_stack = tiff.imread(image_path)

# Patching
tiles_img = patchify(large_image_stack, (640, 640, large_image_stack.shape[2]), step=640)

# Save tiles with 3 bands as PNG
img_idx = 0
for i in range(tiles_img.shape[0]):
    for j in range(tiles_img.shape[1]):
        # Squeeze the img_patch array to remove the extra dimension
        img_patch = tiles_img[i, j, :, :, :].squeeze()

        # Convert to uint8 (if needed, or keep as original dtype)
        img_patch = img_patch.astype(np.uint8)

        # Select only the first three bands (assuming they correspond to RGB)
        img_patch_rgb = img_patch[:, :, :3]  # Select R, G, B bands

        # Save the image as a PNG file
        img = Image.fromarray(img_patch_rgb, 'RGB')
        img.save(f'{tiles_path}/{img_idx}.png')

        img_idx += 1

print(f"Saved {img_idx} tiles to {tiles_path} with RGB format in PNG")


/tmp/ipykernel_2663/3423500399.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img_patch_rgb, 'RGB')


Saved 144 tiles to /content/drive/MyDrive/PROJECT/oilpalm-tree-detection-yolov26/raster_data/tiles/png/3 with RGB format in PNG


## Dataset Anotation
This annotation process aims to assign labels to the tiled data using **Roboflow**.

The general workflow on Roboflow is as follows:

1. Create a new project of type **Object Detection** and upload the tiled images produced from the previous step.
2. Define the object class (in this case, `oilpalm`).
3. Draw bounding boxes around each oil palm crown using the Roboflow annotation interface.
4. Once labeling is complete, assign the dataset into **train**, **validation**, and **test** splits.
5. Generate a dataset version, then download it in **YOLOv26 format** (used in the next notebook for model training).

The annotated dataset can later be downloaded directly into the training environment via the Roboflow API.